# Heart Disease Prediction
## ML Deployment Project

---

## 1. Problem Definition

### ปัญหาคืออะไร?
โรคหัวใจ (Cardiovascular Disease) เป็นสาเหตุการเสียชีวิตอันดับ 1 ของโลก คร่าชีวิตประมาณ 17.9 ล้านคนต่อปี (WHO) การตรวจพบความเสี่ยงตั้งแต่เนิ่นๆ สามารถช่วยให้ผู้ป่วยได้รับการรักษาทันเวลาและลดอัตราการเสียชีวิตได้

### ทำไมถึงน่าแก้ไข?
- การวินิจฉัยโรคหัวใจในปัจจุบันต้องอาศัยการตรวจหลายขั้นตอนที่มีค่าใช้จ่ายสูง
- ML model สามารถช่วย **คัดกรองเบื้องต้น** จากข้อมูลพื้นฐาน เช่น อายุ ความดันโลหิต คอเลสเตอรอล ผล ECG ได้
- ช่วยให้แพทย์มีเครื่องมือสนับสนุนการตัดสินใจ (Decision Support Tool) ไม่ใช่ทดแทนแพทย์

### ทำไม ML ถึงเหมาะกับปัญหานี้?
- เป็น **Binary Classification** ที่ชัดเจน (เป็นโรคหัวใจ / ไม่เป็น)
- มี features ที่เป็นตัวแปรทางการแพทย์ที่มีความสัมพันธ์กับโรคหัวใจตามหลักวิชาการ
- Dataset มีขนาดเพียงพอ (918 ตัวอย่าง) สำหรับ traditional ML
- ความสัมพันธ์ระหว่าง features กับ target อาจเป็น non-linear ซึ่ง ML จัดการได้ดีกว่า rule-based

## 2. Dataset Description

### ที่มาของข้อมูล
Dataset นี้รวบรวมจาก 5 แหล่งข้อมูลโรคหัวใจที่มีชื่อเสียง:
1. Cleveland Clinic Foundation
2. Hungarian Institute of Cardiology
3. V.A. Medical Center, Long Beach
4. University Hospital, Zurich
5. Statlog (Heart) Dataset

เผยแพร่บน [Kaggle](https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction) โดยรวม 918 ตัวอย่างหลังลบข้อมูลซ้ำ

### Features ทั้งหมด 11 ตัว + 1 Target

| Feature | Type | Description | ความหมายทางการแพทย์ |
|---------|------|-------------|---------------------|
| **Age** | Numerical | อายุผู้ป่วย (ปี) | ความเสี่ยงโรคหัวใจเพิ่มขึ้นตามอายุ |
| **Sex** | Categorical (M/F) | เพศ | ผู้ชายมีความเสี่ยงสูงกว่าในช่วงอายุเท่ากัน |
| **ChestPainType** | Categorical | ประเภทอาการเจ็บหน้าอก | ASY (ไม่มีอาการ) พบในผู้ป่วยโรคหัวใจบ่อย |
| **RestingBP** | Numerical | ความดันโลหิตขณะพัก (mm Hg) | ความดันสูง = ปัจจัยเสี่ยง |
| **Cholesterol** | Numerical | คอเลสเตอรอลในเลือด (mg/dl) | ค่าสูงเกินไป = ปัจจัยเสี่ยง |
| **FastingBS** | Binary (0/1) | น้ำตาลในเลือดขณะอดอาหาร > 120 mg/dl | เบาหวานเป็นปัจจัยเสี่ยงโรคหัวใจ |
| **RestingECG** | Categorical | ผลตรวจคลื่นไฟฟ้าหัวใจขณะพัก | Normal / ST-T wave abnormality / LVH |
| **MaxHR** | Numerical | อัตราการเต้นหัวใจสูงสุดที่ทำได้ | ค่าต่ำ = อาจบ่งบอกปัญหาหัวใจ |
| **ExerciseAngina** | Binary (Y/N) | เจ็บหน้าอกขณะออกกำลังกาย | ถ้ามี = สัญญาณเตือนสำคัญ |
| **Oldpeak** | Numerical | ST depression จากการออกกำลังกาย | ค่าสูง = หัวใจขาดเลือด |
| **ST_Slope** | Categorical | ความชันของ ST segment | Flat/Down = ผิดปกติ |
| **HeartDisease** | **Target** (0/1) | 0 = ปกติ, 1 = เป็นโรคหัวใจ | - |

### ประเภท ChestPainType
- **TA** (Typical Angina): เจ็บหน้าอกแบบ classic — เจ็บเมื่อออกแรง หายเมื่อพัก
- **ATA** (Atypical Angina): เจ็บหน้าอกแบบไม่ตรงตำรา
- **NAP** (Non-Anginal Pain): เจ็บหน้าอกที่ไม่เกี่ยวกับหัวใจ
- **ASY** (Asymptomatic): ไม่มีอาการเจ็บหน้าอก — พบบ่อยในผู้ป่วยโรคหัวใจจริงๆ

## 3. Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

In [ ]:
df = pd.read_csv('heart.csv')
print(f'Dataset shape: {df.shape}')
print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')
df.head(10)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# ดู unique values ของ categorical features
cat_cols = ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']
for col in cat_cols:
    print(f'{col}: {df[col].unique()} (n={df[col].nunique()})')

In [ ]:
# ดู class balance ของ target
print('Target Distribution:')
print(df['HeartDisease'].value_counts())
print(f'\nRatio: {df["HeartDisease"].value_counts(normalize=True).round(3).to_dict()}')

## 4. Exploratory Data Analysis (EDA)

### 4.1 Missing Values & ค่าผิดปกติ
จาก dataset นี้ไม่มี null values แต่มี **ค่า 0 ที่น่าสงสัย** — Cholesterol = 0 และ RestingBP = 0 ซึ่งเป็นไปไม่ได้ทางการแพทย์ จึงควรถือว่าเป็น missing values

In [ ]:
# ตรวจสอบ null values
print("Null values per column:")
print(df.isnull().sum())
print(f"\nTotal null values: {df.isnull().sum().sum()}")

# ตรวจสอบค่า 0 ที่น่าสงสัย
print(f"\n--- ค่า 0 ที่ไม่สมเหตุสมผลทางการแพทย์ ---")
print(f"Cholesterol = 0: {(df['Cholesterol'] == 0).sum()} rows ({(df['Cholesterol'] == 0).mean():.1%})")
print(f"RestingBP = 0:   {(df['RestingBP'] == 0).sum()} rows ({(df['RestingBP'] == 0).mean():.1%})")

# Cholesterol = 0 เป็นไปไม่ได้ทางการแพทย์ (คนปกติอยู่ที่ ~125-200 mg/dl)
# RestingBP = 0 ก็เช่นกัน → ถือว่าเป็น missing values

### 4.2 Target Distribution (Class Balance)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

# Count plot
counts = df['HeartDisease'].value_counts()
colors = ['#2ecc71', '#e74c3c']
ax[0].bar(['Normal (0)', 'Heart Disease (1)'], counts.values, color=colors)
ax[0].set_title('Heart Disease Distribution')
ax[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    ax[0].text(i, v + 10, str(v), ha='center', fontweight='bold')

# Pie chart
ax[1].pie(counts.values, labels=['Normal', 'Heart Disease'], autopct='%1.1f%%',
          colors=colors, startangle=90, explode=(0.05, 0.05))
ax[1].set_title('Heart Disease Proportion')

plt.tight_layout()
plt.show()

print(f"Class ratio (Heart Disease / Normal): {counts[1]/counts[0]:.2f}")
print("→ Dataset ค่อนข้าง balanced ไม่จำเป็นต้องทำ oversampling/undersampling")

### 4.3 Distribution ของ Numerical Features

In [ ]:
num_cols = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    ax = axes[i]
    # แยก histogram ตาม HeartDisease
    df[df['HeartDisease'] == 0][col].hist(ax=ax, bins=30, alpha=0.6, color='#2ecc71', label='Normal')
    df[df['HeartDisease'] == 1][col].hist(ax=ax, bins=30, alpha=0.6, color='#e74c3c', label='Heart Disease')
    ax.set_title(f'{col} Distribution')
    ax.set_xlabel(col)
    ax.legend()

# ลบ subplot ที่เหลือ
axes[5].set_visible(False)

plt.suptitle('Numerical Features Distribution by Heart Disease', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Box plots เพื่อดู outliers และเปรียบเทียบระหว่างกลุ่ม
fig, axes = plt.subplots(1, 5, figsize=(20, 5))

for i, col in enumerate(num_cols):
    sns.boxplot(data=df, x='HeartDisease', y=col, ax=axes[i], palette=['#2ecc71', '#e74c3c'])
    axes[i].set_xticklabels(['Normal', 'Heart Disease'])
    axes[i].set_title(col)

plt.suptitle('Box Plots: Numerical Features by Heart Disease', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("สังเกต:")
print("• Age: ผู้ป่วยโรคหัวใจมีอายุเฉลี่ยสูงกว่า")
print("• MaxHR: ผู้ป่วยโรคหัวใจมี max heart rate ต่ำกว่า")
print("• Oldpeak: ผู้ป่วยโรคหัวใจมี ST depression สูงกว่า")
print("• Cholesterol: มี spike ที่ 0 จำนวนมาก (missing values)")

### 4.4 Distribution ของ Categorical Features

In [ ]:
cat_cols = ['Sex', 'ChestPainType', 'FastingBS', 'RestingECG', 'ExerciseAngina', 'ST_Slope']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    # นับจำนวนแยกตาม HeartDisease
    ct = pd.crosstab(df[col], df['HeartDisease'])
    ct.plot(kind='bar', ax=axes[i], color=['#2ecc71', '#e74c3c'], rot=0)
    axes[i].set_title(f'{col} vs Heart Disease')
    axes[i].legend(['Normal', 'Heart Disease'])
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')

plt.suptitle('Categorical Features vs Heart Disease', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("สังเกต:")
print("• Sex: ผู้ชาย (M) มีสัดส่วนเป็นโรคหัวใจสูงกว่าผู้หญิง")
print("• ChestPainType: ASY (ไม่มีอาการ) มีสัดส่วนเป็นโรคหัวใจสูงมาก")
print("• ExerciseAngina: คนที่เจ็บหน้าอกตอนออกกำลังกาย (Y) มีโอกาสเป็นโรคหัวใจสูง")
print("• ST_Slope: Flat มีสัดส่วนเป็นโรคหัวใจสูงมาก, Up มักปกติ")

### 4.5 Correlation Matrix

In [ ]:
# Correlation matrix สำหรับ numerical features
num_df = df[['Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak', 'HeartDisease']]

plt.figure(figsize=(10, 8))
corr = num_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, square=True, linewidths=1)
plt.title('Correlation Matrix (Numerical Features)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Features ที่มี correlation สูงกับ HeartDisease:")
print(corr['HeartDisease'].drop('HeartDisease').sort_values(ascending=False).to_string())
print("\nสังเกต:")
print("• Oldpeak มี positive correlation สูงสุดกับ HeartDisease")
print("• MaxHR มี negative correlation สูงสุด (ยิ่ง MaxHR ต่ำ ยิ่งเสี่ยง)")
print("• Age มี positive correlation ปานกลาง")

### 4.6 สรุป EDA

**สิ่งที่ค้นพบ:**
1. **Missing values ซ่อนอยู่**: Cholesterol = 0 (~172 rows) และ RestingBP = 0 (1 row) → ต้องจัดการใน preprocessing
2. **Class balance**: ค่อนข้าง balanced (~55% Heart Disease, ~45% Normal) → ไม่จำเป็นต้อง resample
3. **Features ที่ส่งสัญญาณชัด**: ST_Slope (Flat), ExerciseAngina (Y), ChestPainType (ASY), Oldpeak สูง, MaxHR ต่ำ
4. **Cholesterol กับ RestingBP** มี correlation ต่ำกับ target — อาจเพราะ missing values (0) ทำให้ signal หาย

**แนวทาง Preprocessing:**
- แทนค่า 0 ใน Cholesterol และ RestingBP ด้วย median (แยกตาม group ถ้าเป็นไปได้)
- ใช้ Pipeline เพื่อรวม preprocessing + model เข้าด้วยกัน ป้องกัน data leakage

## 5. Data Preprocessing & Pipeline

### แนวทาง:
1. แทนค่า 0 ใน Cholesterol และ RestingBP ด้วย NaN แล้วใช้ median imputation
2. Numerical features → StandardScaler
3. Categorical features → OneHotEncoder
4. รวมทั้งหมดเข้า Pipeline เพื่อป้องกัน data leakage

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, f1_score)

# แยก features กับ target
X = df.drop('HeartDisease', axis=1).copy()
y = df['HeartDisease'].copy()

# แทนค่า 0 ที่เป็น missing values ด้วย NaN
X.loc[X['Cholesterol'] == 0, 'Cholesterol'] = np.nan
X.loc[X['RestingBP'] == 0, 'RestingBP'] = np.nan

print(f"After replacing 0s with NaN:")
print(f"  Cholesterol NaN: {X['Cholesterol'].isna().sum()}")
print(f"  RestingBP NaN:   {X['RestingBP'].isna().sum()}")

# แบ่ง train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain set: {X_train.shape[0]} rows")
print(f"Test set:  {X_test.shape[0]} rows")

In [ ]:
# กำหนด column types
numerical_features = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']
categorical_features = ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']

# สร้าง preprocessing pipeline
numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),  # เติม NaN ด้วย median
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('encoder', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numerical_pipeline, numerical_features),
    ('cat', categorical_pipeline, categorical_features)
])

print("Preprocessing pipeline created!")
print(f"  Numerical ({len(numerical_features)}): {numerical_features}")
print(f"  Categorical ({len(categorical_features)}): {categorical_features}")

## 6. Baseline Model — Logistic Regression

เริ่มจาก Logistic Regression เป็น baseline เพราะ:
- เป็น model ง่ายที่ interpretable
- ทำงานได้ดีกับ binary classification
- ใช้เป็น benchmark เปรียบเทียบกับ model อื่น

In [ ]:
from sklearn.linear_model import LogisticRegression

# สร้าง full pipeline: preprocessing + model
baseline_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

# Train
baseline_pipeline.fit(X_train, y_train)

# Predict
y_pred = baseline_pipeline.predict(X_test)
y_prob = baseline_pipeline.predict_proba(X_test)[:, 1]

# Evaluation
print("=" * 50)
print("BASELINE MODEL: Logistic Regression")
print("=" * 50)
print(f"\nAccuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred):.4f}")
print(f"ROC AUC:   {roc_auc_score(y_test, y_prob):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Heart Disease']))

In [ ]:
# Confusion Matrix + ROC Curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Normal', 'Heart Disease'],
            yticklabels=['Normal', 'Heart Disease'])
axes[0].set_title('Confusion Matrix')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)
axes[1].plot(fpr, tpr, color='#e74c3c', lw=2, label=f'Logistic Regression (AUC = {auc:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Random (AUC = 0.500)')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1.05])

plt.tight_layout()
plt.show()

In [ ]:
# Cross-validation เพื่อดูว่า model stable แค่ไหน
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(baseline_pipeline, X_train, y_train, cv=cv, scoring='accuracy')

print("5-Fold Cross-Validation Results:")
print(f"  Scores: {cv_scores.round(4)}")
print(f"  Mean:   {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"\n→ Model ค่อนข้าง stable (std ต่ำ) — พร้อมเปรียบเทียบกับ model อื่น")

## 7. Model Comparison

เปรียบเทียบหลาย algorithms เพื่อหา model ที่ดีที่สุด:
- Logistic Regression (baseline)
- Random Forest
- Gradient Boosting
- Support Vector Machine (SVM)
- K-Nearest Neighbors (KNN)

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# สร้าง models
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42, n_estimators=100),
    'SVM': SVC(random_state=42, probability=True),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}

# เปรียบเทียบด้วย cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

print(f"{'Model':<25} {'Accuracy':<12} {'F1':<12} {'ROC AUC':<12}")
print("=" * 61)

for name, model in models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    
    acc_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='accuracy')
    f1_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='f1')
    auc_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc')
    
    results[name] = {
        'accuracy': acc_scores.mean(),
        'accuracy_std': acc_scores.std(),
        'f1': f1_scores.mean(),
        'f1_std': f1_scores.std(),
        'roc_auc': auc_scores.mean(),
        'roc_auc_std': auc_scores.std()
    }
    
    print(f"{name:<25} {acc_scores.mean():.4f}±{acc_scores.std():.3f} "
          f"{f1_scores.mean():.4f}±{f1_scores.std():.3f} "
          f"{auc_scores.mean():.4f}±{auc_scores.std():.3f}")

In [ ]:
# Visualize comparison
results_df = pd.DataFrame(results).T

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
metrics = ['accuracy', 'f1', 'roc_auc']
titles = ['Accuracy', 'F1 Score', 'ROC AUC']

for i, (metric, title) in enumerate(zip(metrics, titles)):
    ax = axes[i]
    bars = ax.barh(results_df.index, results_df[metric], 
                   xerr=results_df[f'{metric}_std'], color='#3498db', capsize=5)
    ax.set_title(title)
    ax.set_xlim(0.75, 1.0)
    
    for bar, val in zip(bars, results_df[metric]):
        ax.text(val + 0.005, bar.get_y() + bar.get_height()/2, f'{val:.4f}',
                va='center', fontweight='bold', fontsize=10)

plt.suptitle('Model Comparison (5-Fold CV)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Hyperparameter Tuning

เลือก **Gradient Boosting** (หรือ model ที่ได้คะแนนสูงสุด) มาทำ hyperparameter tuning ด้วย GridSearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV

# Gradient Boosting Pipeline
gb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', GradientBoostingClassifier(random_state=42))
])

# Parameter grid
param_grid = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__learning_rate': [0.05, 0.1, 0.2],
    'classifier__max_depth': [3, 4, 5],
    'classifier__min_samples_split': [2, 5],
    'classifier__subsample': [0.8, 1.0]
}

# GridSearchCV
grid_search = GridSearchCV(
    gb_pipeline, param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"\nBest Parameters:")
for param, value in grid_search.best_params_.items():
    print(f"  {param.replace('classifier__', '')}: {value}")
print(f"\nBest ROC AUC (CV): {grid_search.best_score_:.4f}")

In [ ]:
# Evaluate best model on test set
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)
y_prob_best = best_model.predict_proba(X_test)[:, 1]

print("=" * 50)
print("BEST MODEL: Tuned Gradient Boosting")
print("=" * 50)
print(f"\nAccuracy:  {accuracy_score(y_test, y_pred_best):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred_best):.4f}")
print(f"ROC AUC:   {roc_auc_score(y_test, y_prob_best):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_best, target_names=['Normal', 'Heart Disease']))

In [ ]:
# Confusion Matrix + ROC Curve ของ best model
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', ax=axes[0],
            xticklabels=['Normal', 'Heart Disease'],
            yticklabels=['Normal', 'Heart Disease'])
axes[0].set_title('Confusion Matrix — Tuned Gradient Boosting')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# ROC Curve — เปรียบเทียบ baseline vs best
fpr_base, tpr_base, _ = roc_curve(y_test, baseline_pipeline.predict_proba(X_test)[:, 1])
fpr_best, tpr_best, _ = roc_curve(y_test, y_prob_best)

axes[1].plot(fpr_base, tpr_base, color='#3498db', lw=2, 
             label=f'Logistic Regression (AUC = {roc_auc_score(y_test, baseline_pipeline.predict_proba(X_test)[:, 1]):.3f})')
axes[1].plot(fpr_best, tpr_best, color='#e74c3c', lw=2,
             label=f'Tuned GB (AUC = {roc_auc_score(y_test, y_prob_best):.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve Comparison')
axes[1].legend()

plt.tight_layout()
plt.show()

## 9. Feature Importance

In [ ]:
# ดึง feature names หลัง OneHotEncoding
ohe_features = best_model.named_steps['preprocessor'].named_transformers_['cat'] \
    .named_steps['encoder'].get_feature_names_out(categorical_features).tolist()
all_features = numerical_features + ohe_features

# Feature importance จาก Gradient Boosting
importances = best_model.named_steps['classifier'].feature_importances_
feat_imp = pd.Series(importances, index=all_features).sort_values(ascending=True)

plt.figure(figsize=(10, 8))
feat_imp.plot(kind='barh', color='#e74c3c')
plt.title('Feature Importance — Tuned Gradient Boosting', fontsize=14, fontweight='bold')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print("Top 5 Features:")
for feat, imp in feat_imp.sort_values(ascending=False).head(5).items():
    print(f"  {feat}: {imp:.4f}")

## 10. Save Model

บันทึก model ที่ดีที่สุดเป็น `.pkl` เพื่อนำไป deploy ใน Streamlit app

In [ ]:
import joblib

# Save best model
joblib.dump(best_model, 'heart_disease_model.pkl')
print("Model saved: heart_disease_model.pkl")

# Verify
loaded_model = joblib.load('heart_disease_model.pkl')
y_pred_verify = loaded_model.predict(X_test)
print(f"Verification accuracy: {accuracy_score(y_test, y_pred_verify):.4f}")
print("→ Model saved and verified successfully!")